# 1. Definição do Problema e Carga de Dados
Desenvolver um modelo preditivo para prever a evasão de curso técnico de um aluno na UFPB. Vamos seguir a lógica já estabelecida e incrementá-la com pipelines mais robustos, XGBoost e explicabilidade com SHAP.

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore') # Filter warnings for cleaner output

datadir = 'dados' # Pasta com os dados corrigida

# 1.1 Carga das Tabelas
discentes = pd.read_parquet(os.path.join(datadir, 'discentes.parquet'))
discentes.drop_duplicates(subset=['id_discente'], inplace=True, ignore_index=True)

matriculas = pd.read_parquet(os.path.join(datadir, 'matriculas.parquet'))
componentes = pd.read_parquet(os.path.join(datadir, 'componentes.parquet'))
cursos = pd.read_parquet(os.path.join(datadir, 'cursos.parquet'))
docentes = pd.read_parquet(os.path.join(datadir, 'docentes.parquet'))

print(f"Cursos: {len(cursos)}, Discentes: {len(discentes)}, Matrículas: {len(matriculas)}")


Cursos: 1310, Discentes: 7705, Matrículas: 426753


In [2]:
# 1.2 Criação da Tabela Única (Merge)
status_df = (
    matriculas.merge(discentes, how='left', on='id_discente')
    .merge(componentes, how='left', on='id_disciplina')
    .merge(cursos, how='left', on=['id_curso', 'id_estrutura_curricular', 'id_disciplina'], suffixes=['', '_curso'])
)


## 2. Engenharia de Variáveis (Feature Engineering)
Removendo **Vazamento de Dados (Data Leakage)**: Computando o histórico apenas para o primeiro período letivo do aluno e removendo variáveis futuras (como carga horária final integralizada) que "entregavam" a resposta para os modelos.

In [3]:
# 2.1 Desempenho Histórico Evitando Vazamento
# Para prever a evasão ainda no início do curso (ex: final do 1º semestre),
# usaremos apenas os dados do primeiro período letivo do aluno.

print("Filtrando apenas o 1º período letivo para evitar Data Leakage...")
primeiro_periodo_df = status_df.groupby('id_discente')[['ano', 'periodo']].min().reset_index()
status_1o_semestre = status_df.merge(primeiro_periodo_df, on=['id_discente', 'ano', 'periodo'], how='inner')

hist_disc = (
    status_1o_semestre[['situacao', 'id_disciplina', 'id_discente']]
    .assign(
        aprov_1o_sem = lambda x: x.situacao.str.contains("^APROV", na=False).astype(int),
        reprov_1o_sem = lambda x: x.situacao.str.contains("^REP", na=False).astype(int)
    )
    .groupby(['id_discente'])
    .agg({"aprov_1o_sem": "sum", "reprov_1o_sem": "sum"}) 
    .reset_index()
)

# 2.2 Target e Variáveis Base
df = status_df.copy()
df = df.dropna(subset=['status_discente'])

# Target: Evasão = 1 (CANCELADO)
df["evasao"] = (df["status_discente"] == "CANCELADO").astype(int)

# Variáveis (No momento da matrícula ou do 1º semestre)
df["idade_ingresso"] = df["ano_ingresso"] - df["ano_nascimento"]
df["ingresso_recente"] = (df["ano_ingresso"] >= 2020).astype(int)

# 2.3 Junção das Features da Dimensão Discente
x1 = (df
    .dropna(subset=['status_discente', 'idade_ingresso'])
    .drop_duplicates(subset='id_discente', ignore_index=True)
    .merge(hist_disc, on='id_discente', how='left')
    .astype({"id_curso": int})
)

# Preencher NAs (Alunos sem notas no 1º semestre) com 0
x1['aprov_1o_sem'] = x1['aprov_1o_sem'].fillna(0)
x1['reprov_1o_sem'] = x1['reprov_1o_sem'].fillna(0)

Filtrando apenas o 1º período letivo para evitar Data Leakage...


In [4]:
# 2.4 Criação de Dummies
def gera_dummies(df, colname, map_dict=None):
    if map_dict is not None:
        df[colname] = df[colname].map(map_dict)
    else:
        df[colname] = df[colname].astype(str).str.lower()
    return pd.get_dummies(df, columns=[colname], drop_first=True, dtype=int)
    
x1 = gera_dummies(x1, 'sexo',  {"F": "feminino", "M": "masculino"})
x1 = gera_dummies(x1, 'estado_civil',  {"Solteiro(a)": "solteiro", "Casado(a)": "casado", "Outro": "outro"})

for col in ['raca_declarada', 'faixa_renda_familiar', 'id_curso']:
    x1 = gera_dummies(x1, col)

# Seleção de Colunas Finais (Removidas ch_integralizada e progresso para evitar Leakage)
colunas_base = ['evasao', 'ano_ingresso', 'periodo_ingresso', 'idade_ingresso',  
                'aprov_1o_sem', 'reprov_1o_sem', 'ingresso_recente']

colunas_dummies = [c for c in x1.columns if c.startswith(("sexo_", "estado_civil_", "raca_declarada_", "faixa_renda_familiar_", "id_curso_"))]

data = x1[colunas_base + colunas_dummies]

## 3. Separação de Treino e Teste (Split Temporal)
Mantendo a estratégia lógica que já havia sido definida: O corte temporal reflete cenários reais de predição para cohortes futuras.

In [5]:
# Mantendo a premissa definida:
CUTOFF = 2020

# Excluir os dois ultimos anos de ingressantes para evitar problema do tempo de curso
ANO_ATUAL = data.ano_ingresso.max()
data = data[data["ano_ingresso"] <= ANO_ATUAL - 2]

train = data[data["ano_ingresso"] <= CUTOFF].copy()
test  = data[data["ano_ingresso"] > CUTOFF].copy()

print(f"Treino: {len(train)} linhas ({len(train)/len(data):.1%}) | Evasão: {train['evasao'].mean():.1%}")
print(f"Teste: {len(test)} linhas ({len(test)/len(data):.1%}) | Evasão: {test['evasao'].mean():.1%}")

X_train = train.drop(['evasao'], axis=1)
y_train = train['evasao']
X_test = test.drop(['evasao'], axis=1)
y_test = test['evasao']


Treino: 3363 linhas (59.9%) | Evasão: 55.0%
Teste: 2250 linhas (40.1%) | Evasão: 47.0%


## 4. Modelagem Sênior (Optuna, Calibration & Recall MLOps)
Evoluindo a busca em Grade para **Otimização Bayesiana** (Maximização Dinâmica do F1/PR-AUC), e adoção da calibração isotônica de probabilidades.

In [ ]:
import optuna
import shap
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, f1_score, RocCurveDisplay, PrecisionRecallDisplay, roc_auc_score, average_precision_score

print("Iniciando Otimização Bayesiana com Optuna (Foco em PR-AUC para lidar com classe minoritária Evasão)...")

def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
    }
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]
        
        scale_weight = (len(y_tr) - sum(y_tr)) / sum(y_tr)
        model = XGBClassifier(**param, random_state=42, eval_metric="logloss", scale_pos_weight=scale_weight)
        model.fit(X_tr, y_tr)
        
        preds = model.predict_proba(X_va)[:, 1]
        scores.append(average_precision_score(y_va, preds))
        
    return np.mean(scores)

study = optuna.create_study(direction='maximize')
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=15) # Pequeno para tempo rápido em teste
best_params = study.best_params
print(f"\nHiperparâmetros Vencedores Optuna: {best_params}")

# Treino Final + Calibração Estatística
print("Treinando Novo Algoritmo e Calibrando Probabilidades...")
scale_w = (len(y_train) - sum(y_train)) / sum(y_train)
best_xgb = XGBClassifier(**best_params, random_state=42, eval_metric="logloss", scale_pos_weight=scale_w)
calibrated_xgb = CalibratedClassifierCV(best_xgb, method='isotonic', cv=5)
calibrated_xgb.fit(X_train, y_train)

# Encontrar Threshold Focado em Recall da classe Evasão
prob_train = calibrated_xgb.predict_proba(X_train)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_train, prob_train)
valid_thresholds = [t for p, r, t in zip(precision, recall, thresholds) if r >= 0.88]
best_threshold = valid_thresholds[-1] if valid_thresholds else 0.5
print(f"Threshold Estratégico para Minimizar Falsos Negativos (Recall > 88%): {best_threshold:.4f}\n")


## 5. Validação com Visão de Negócio
Métricas no Teste Out-of-Time com Matriz de Confusão ponderada pelo custo financeiro/pedagógico de errar.

In [ ]:
prob_test = calibrated_xgb.predict_proba(X_test)[:, 1]
pred_test = (prob_test >= best_threshold).astype(int)

print("---- PERFORMANCE AFERIDA OUT-OF-TIME ----")
print(f"PR-AUC (Area da Precision-Recall - Ótima p/ raros): {average_precision_score(y_test, prob_test):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, prob_test):.4f}\n")
print(classification_report(y_test, pred_test))

fig, ax = plt.subplots(1, 3, figsize=(18, 5))
RocCurveDisplay.from_predictions(y_test, prob_test, ax=ax[0], color='darkorange')
ax[0].set_title("Curva Curva ROC Clássica")
ax[0].plot([0,1],[0,1], 'k--')

PrecisionRecallDisplay.from_predictions(y_test, prob_test, ax=ax[1], color='darkorange')
ax[1].set_title("Curva PR (A Melhor para Evasão)")

cf_mat = confusion_matrix(y_test, pred_test)
sns.heatmap(cf_mat, annot=True, fmt='d', cmap='Oranges', ax=ax[2])
ax[2].set_title("Matriz de Custo vs Erro\n(O Modelo prioriza resgatar alunos evitando 0's na diag sup.)")
ax[2].set_ylabel('Rótulo Verdadeiro (1=Evasão)')
ax[2].set_xlabel('Previsão Sistêmica (Disparo de Ação)')

plt.tight_layout()
plt.show()


## 6. SHAP Independence Plot & Thresholds Prescritivos
Mostrando visualmente aos coordenadores como as Faltas e Reprovações disparam O RISCO de Evasão de forma microscópica.

In [ ]:
best_xgb.fit(X_train, y_train)
explainer = shap.TreeExplainer(best_xgb)
shap_values = explainer(X_test)

print("A) Visão Executiva Macro das Causas Raízes:")
shap.plots.beeswarm(shap_values, max_display=7)

print("\nB) Gráfico Insight (Dependence Plot):")
print("A partir de qual nr exato de reprovações o log-odds estoura como risco crítico?")
try:
    shap.plots.scatter(shap_values[:, "reprov_1o_sem"], color=shap_values[:, "idade_ingresso"])
except Exception as e:
    print(f"Dependence ignorado para variável - {e}")
